# 04 Batch and parallel extraction

`extract_features_from_folder` and the `afx` command line run one config over every audio file below a folder and write one table. This notebook covers output formats, failed files, `n_jobs` and the CLI.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd

import afx_audio as afx
import demo_data

SR = 16000
tmp = Path(tempfile.mkdtemp(prefix='afx_batch_'))
audio_dir = tmp / 'audio'

## A folder of clips

Up to 24 files from the configured dataset, otherwise 24 synthetic clips in one subfolder per class. One file is deliberately unreadable.

In [ ]:
files = demo_data.find_audio(max_files=24)
if files:
    audio_dir.mkdir()
    for f in files:
        shutil.copy(f, audio_dir / os.path.basename(f))
else:
    demo_data.write_synthetic_folder(str(audio_dir), n_per_class=6, seconds=2.0, sr=SR)
(audio_dir / 'broken.wav').write_bytes(b'not a wav file')

n_files = sum(1 for p in audio_dir.rglob('*.wav'))
print(f'{n_files} .wav files under {audio_dir}')

## One call, one table

The config can be a dict (or a preset), a JSON path, or the bundled default. Keys of the result are paths relative to the folder, with forward slashes on every OS.

In [ ]:
config = {
    'sample_rate': SR,
    'features': {
        'mfcc': {'n_mfcc': 13, 'frame_size': 1024, 'hop_length': 512},
        'spectral_centroid': {'frame_size': 1024, 'hop_length': 512},
        'spectral_flatness': {'frame_size': 1024, 'hop_length': 512},
        'zcr': {'frame_size': 1024, 'hop_length': 512},
        'hnr': {'frame_size': 1024, 'hop_length': 512},
    },
    'aggregation': 'mean',
}
results = afx.extract_features_from_folder(str(audio_dir), config, str(tmp / 'features.csv'), verbose=False)
ok = [k for k in results if k != 'errors']
print(len(ok), 'files extracted; first key:', ok[0])
print('errors:', results.get('errors'))

## Failed files

A file that cannot be read or processed never stops the batch. Its error goes to `<output>.errors.json` and to the `'errors'` entry of the returned dict, and is never mixed into the feature table.

In [ ]:
print(json.loads((tmp / 'features.csv.errors.json').read_text()))

## Output formats

The format follows the output extension (`save_format=` or `afx --format` override it).

| extension | content | load with |
|---|---|---|
| `.csv` | one row per file, named columns | `pd.read_csv(path, index_col='file')` |
| `.parquet` | same as CSV, typed; needs `afx-audio[parquet]` | `pd.read_parquet(path)` |
| `.json` | `{file: {feature: list}}`, NaN as `null` | `json.load` |
| `.npy` | pickled dict of arrays | `np.load(path, allow_pickle=True).item()` |

In [ ]:
import warnings

warnings.filterwarnings('ignore', message='1 file')  # broken.wav, reported above
for ext in ('csv', 'parquet', 'json', 'npy'):
    afx.extract_features_from_folder(str(audio_dir), config, str(tmp / f'features.{ext}'), verbose=False)

df = pd.read_csv(tmp / 'features.csv', index_col='file')
pq = pd.read_parquet(tmp / 'features.parquet')
js = json.loads((tmp / 'features.json').read_text())
npy = np.load(tmp / 'features.npy', allow_pickle=True).item()

print('csv    ', df.shape, list(df.columns[:3]), '...')
print('parquet', pq.shape)
print('json   ', len(js), 'files;', {k: len(v) for k, v in next(iter(js.values())).items()})
print('npy    ', len(npy), 'files;', {k: len(v) for k, v in next(iter(npy.values())).items()})
np.testing.assert_allclose(df.to_numpy(), pq.loc[df.index].to_numpy())

Column names match `afx.flatten_features` and `AFXFeatureExtractor.get_feature_names_out()`, so a table written here lines up with features computed in memory.

In [ ]:
signal, sr = afx.load_audio(audio_dir / df.index[0], sr=SR)
names, vector = afx.flatten_features(afx.extract_all_features(signal, sr, config))
assert names == list(df.columns)
np.testing.assert_allclose(vector, df.iloc[0].to_numpy(), equal_nan=True)
df.head()

## Serial and parallel

`n_jobs=N` processes files in `N` worker processes (`0` or negative means all cores). Output and order are identical to a serial run. Worker start-up is a fixed cost, so the speed-up approaches the core count only for long files or many of them. In a script, call it under `if __name__ == '__main__':` (see `03_long_recordings_and_custom_features.ipynb`).

In [ ]:
timings = {}
outputs = {}
for n_jobs in (1, 2):
    t0 = time.perf_counter()
    outputs[n_jobs] = afx.extract_features_from_folder(
        str(audio_dir), config, str(tmp / f'features_j{n_jobs}.npy'), n_jobs=n_jobs, verbose=False)
    timings[n_jobs] = time.perf_counter() - t0

assert list(outputs[1]) == list(outputs[2])
for key, feats in outputs[1].items():
    if key == 'errors':
        assert feats == outputs[2]['errors']
        continue
    for name, value in feats.items():
        np.testing.assert_array_equal(value, outputs[2][key][name])
print({f'n_jobs={k}': f'{v:.2f} s' for k, v in timings.items()}, f'on {os.cpu_count()} cores; outputs identical')

## The `afx` command

The installed `afx` command runs the same function. It is invoked here as `python -m afx_audio.cli` so the notebook uses the kernel's interpreter; in a shell write `afx ...`.

In [ ]:
def afx_cli(*args):
    run = subprocess.run([sys.executable, '-m', 'afx_audio.cli', *args], capture_output=True, text=True)
    print(f'$ afx {" ".join(args)}  (exit {run.returncode})')
    print((run.stdout + run.stderr).strip()[:1500], '\n')


afx_cli('--describe', 'spectral_flatness')
afx_cli(str(audio_dir), '--preset', 'speech', '-o', str(tmp / 'speech.parquet'), '-q')
afx_cli(str(audio_dir), '--features', 'mfcc,zcr', '--aggregation', 'summary', '-o', str(tmp / 'small.csv'), '-q')
print(pd.read_parquet(tmp / 'speech.parquet').shape, pd.read_csv(tmp / 'small.csv').columns[:5].tolist())

`-j N` sets `n_jobs`. `afx --help` lists every option; `--list-features` and `--list-presets` print the catalogues.

In [ ]:
shutil.rmtree(tmp)